<a href="https://colab.research.google.com/github/ZaraFatima123/AQI_DEMO/blob/main/sign_language_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**[Teaching Machines to Understand Sign Language](https://www.sciencebuddies.org/)**

This notebook was developed by Science Buddies [www.sciencebuddies.org](https://www.sciencebuddies.org/) as part of a science project to allow students to explore and learn about artificial intelligence. For personal use, this notebook can be downloaded and modified with attribution. For all other uses, please see our [Terms and Conditions of Fair Use](https://www.sciencebuddies.org/about/terms-and-conditions-of-fair-use).  

**Troubleshooting tips**
*   Read the written instructions at Science Buddies and the text and comments on this page carefully.
*   If you make changes that break the code, you can download a fresh copy of this notebook and start over.

*   If you are using this notebook for a science project and need help, visit our [Ask an Expert](https://www.sciencebuddies.org/science-fair-projects/ask-an-expert-intro) forum for assistance.

## **How To Use This Notebook**

This notebook contains text fields, like this one, that give you information about the project and instructions.

In [ ]:
# There are also code blocks, like this one.

# The green text in a code block are comments. Comments are descriptions of what the code does.

# The non-green text in a code block is the Python code. Click on the triangle in the top left corner to run this code block.

print("Congratulations, you ran a code block! Try changing the text in the code and running it again.")

Congratulations, you ran a code block! Try changing the text in the code and running it again.


# Installing Dependencies

In [ ]:
!pip uninstall -y mediapipe
!pip install -U "numpy<2" "protobuf<5"
!pip install --no-cache-dir "mediapipe==0.10.21"

Found existing installation: mediapipe 0.10.21
Uninstalling mediapipe-0.10.21:
  Successfully uninstalled mediapipe-0.10.21
  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.0 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.4.2
    Uninstalling numpy-2.4.2:
      Successfully uninstalled numpy-2.4.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grain 0.2.15 requires protobuf>=5.28.3, but you have protobuf 4.25.8 which is incompatible.
pytensor 2.37.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.50.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.90 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 wh

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.6/35.6 MB 44.4 MB/s eta 0:00:00


In [ ]:
# This command installs the TensorFlow library using pip (Python's package installer)
!pip install tensorflow

In [ ]:
# Install the OpenCV library, which is used for computer vision tasks such as image and video processing.
!pip install opencv-python

  Using cached numpy-2.4.2-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
Using cached numpy-2.4.2-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.6 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mediapipe 0.10.21 requires numpy<2, but you have numpy 2.4.2 which is incompatible.
grain 0.2.15 requires protobuf>=5.28.3, but you have protobuf 4.25.8 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.2 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.2 which is incompatible.
ydf 0.14.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.8 which is incompatible.


# 1. Importing Libraries

In [ ]:
# General libraries
import os  # For interacting with the operating system (e.g., file handling)
import time  # For time-related functions
import numpy as np  # For numerical operations and handling arrays
import scipy.stats  # For statistical functions (e.g., mode, z-test)
import random
import pandas as pd

# Visualization libraries
import matplotlib.pyplot as plt  # For plotting graphs and images
from IPython.display import display, Image  # For displaying images and other media in Jupyter/Colab

# Computer Vision Libraries
import cv2  # For computer vision tasks such as image and video processing
import mediapipe as mp  # For using MediaPipe's machine learning models for pose detection and more
from google.colab.patches import cv2_imshow  # For displaying images with OpenCV in Google Colab

# Machine Learning Libraries
from sklearn.model_selection import train_test_split  # For splitting data into training and testing sets
from sklearn.metrics import multilabel_confusion_matrix, accuracy_score  # For model evaluation

# Deep Learning Libraries (TensorFlow and Keras)
from tensorflow.keras.utils import to_categorical  # For converting labels to categorical format (one-hot encoding)
from tensorflow.keras.models import Sequential  # For defining the neural network architecture
from tensorflow.keras.layers import LSTM, Dense  # For adding LSTM and Dense layers to the model
from tensorflow.keras.callbacks import TensorBoard  # For logging training progress for TensorBoard
from tensorflow.keras.models import load_model  # For loading pre-trained models

In [ ]:
# Import the 'drive' module from Google Colab to interact with Google Drive
from google.colab import drive

# Mount Google Drive to access files stored in it. This will create a folder 'drive' in the '/content' directory.
# After running this, you will be prompted to authorize Colab to access your Google Drive.
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
# Set the base directory to the specified path on Google Drive.
# This is the location where the 'sign_language_detection' folder is stored within the 'MyDrive' folder.
base_dir = "/content/drive/MyDrive/sign_language_detection"

test_videos_path = os.path.join(base_dir, 'test_videos')
if not os.path.exists(test_videos_path):
    os.makedirs(test_videos_path)

DATA_PATH = os.path.join(base_dir, 'MP_Data')

# 2. Detecting Keypoints using MP Holistic

Code Block 2A

In [ ]:
# Initialize the Holistic model from MediaPipe for full-body pose detection.
# This model can detect landmarks for the face, hands, and body.
mp_holistic = mp.solutions.holistic

# Initialize the drawing utilities from MediaPipe to draw the landmarks on images or videos.
# These utilities help visualize the pose detection results by drawing the keypoints and connections.
mp_drawing = mp.solutions.drawing_utils

# This function processes an image using a MediaPipe model to detect landmarks/poses and returns the processed image and the results.
def mediapipe_detection(image, model):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) # COLOR CONVERSION BGR 2 RGB
    image.flags.writeable = False                  # Image is no longer writeable
    results = model.process(image)                 # Make prediction
    image.flags.writeable = True                   # Image is now writeable
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR) # COLOR COVERSION RGB 2 BGR
    return image, results

# This function draws the detected pose, left hand, and right hand landmarks with their connections on the given image.
def draw_landmarks(image, results):
    mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS) # Draw pose connections
    mp_drawing.draw_landmarks(image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS) # Draw left hand connections
    mp_drawing.draw_landmarks(image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS) # Draw right hand connections

# This function draws the pose, left hand, and right hand landmarks with custom styling (color, thickness, and circle radius)
# on the given image, using the MediaPipe Holistic model's results.
def draw_styled_landmarks(image, results):
    # Draw pose connections
    mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS,
                             mp_drawing.DrawingSpec(color=(80,22,10), thickness=2, circle_radius=4),
                             mp_drawing.DrawingSpec(color=(80,44,121), thickness=2, circle_radius=2)
                             )
    # Draw left hand connections
    mp_drawing.draw_landmarks(image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS,
                             mp_drawing.DrawingSpec(color=(121,22,76), thickness=2, circle_radius=4),
                             mp_drawing.DrawingSpec(color=(121,44,250), thickness=2, circle_radius=2)
                             )
    # Draw right hand connections
    mp_drawing.draw_landmarks(image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS,
                             mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=4),
                             mp_drawing.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=2)
                             )

Code Block 2B

In [ ]:
# TODO: Change the name of the test video to ensure the code is running
video_name = "new1.mp4"
video_path = os.path.join(test_videos_path, video_name)

# Try to open the video
cap = cv2.VideoCapture(video_path)

# Check if the video opened successfully
if not cap.isOpened():
    raise FileNotFoundError(f"Error: Cannot open video file '{video_path}'. Please check if the file exists and the path is correct.")

# Set MediaPipe model
mp_holistic = mp.solutions.holistic
with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
    while cap.isOpened():
        ret, frame = cap.read()
        frame = cv2.flip(frame, 0)

        if not ret:
            break

        # ✅ Flip ONCE at the very beginning (fixes input video orientation)
        frame = cv2.flip(frame, 0)

        # Make detections
        image, results = mediapipe_detection(frame, holistic)

        # Draw landmarks on the frame
        draw_styled_landmarks(image, results)

        # Convert frame to RGB (to display in Colab)
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # Display the frame
        plt.figure(figsize=(10, 10))
        plt.imshow(image_rgb)
        plt.axis('off')
        display(plt.gcf())
        plt.close()

        # Pause for a short moment to simulate frame rate
        cv2.waitKey(1)

    cap.release()

# 3. Extracting Keypoint Values

Code Block 3A

In [ ]:
import numpy as np

def extract_keypoints(results):
    """
    Extracts pose, face, left hand, and right hand landmarks from MediaPipe results
    and returns a single flattened NumPy array.
    """

    # Pose: 33 landmarks × (x, y, z, visibility)
    pose = np.array(
        [[res.x, res.y, res.z, res.visibility]
         for res in results.pose_landmarks.landmark]
    ).flatten() if results.pose_landmarks else np.zeros(33 * 4)

    # Face: 468 landmarks × (x, y, z)
    face = np.array(
        [[res.x, res.y, res.z]
         for res in results.face_landmarks.landmark]
    ).flatten() if results.face_landmarks else np.zeros(468 * 3)

    # Left Hand: 21 landmarks × (x, y, z)
    lh = np.array(
        [[res.x, res.y, res.z]
         for res in results.left_hand_landmarks.landmark]
    ).flatten() if results.left_hand_landmarks else np.zeros(21 * 3)

    # Right Hand: 21 landmarks × (x, y, z)
    rh = np.array(
        [[res.x, res.y, res.z]
         for res in results.right_hand_landmarks.landmark]
    ).flatten() if results.right_hand_landmarks else np.zeros(21 * 3)

    # Concatenate all keypoints into one vector
    return np.concatenate([pose, face, lh, rh])

# 4. Setting up Folders for Collection

Code Block 4A

In [ ]:
# TODO: Define the set of actions (sign language gestures) to detect
actions = np.array(['hello', 'thank_you', 'see_you_later'])

# Path for exported data (numpy arrays will be saved here)
DATA_PATH = os.path.join(base_dir, 'MP_Data')
VIDEO_PATH = os.path.join(base_dir, 'videos')

# Create the data directory if it doesn't exist
if not os.path.exists(DATA_PATH):
    os.makedirs(DATA_PATH)

# Number of sequences (videos) to record per action
no_sequences = 30

# Number of frames per video sequence
sequence_length = 30

# Starting index for naming folders that store each sequence
start_folder = 30

# Create folder structure for each action and each sequence
for action in actions:
    # Path for this action's data
    action_path = os.path.join(DATA_PATH, action)
    video_path = os.path.join(VIDEO_PATH, action)

    # Create action folder if it doesn't exist
    if not os.path.exists(action_path):
        os.makedirs(action_path)

    # Create video folder if it doesn't exist
    if not os.path.exists(video_path):
        os.makedirs(video_path)

    # Find existing numbered folders (sequences) for this action
    existing_dirs = [d for d in os.listdir(action_path) if d.isdigit()]

    # Determine the current max sequence number to avoid overwriting
    if existing_dirs:
        dirmax = np.max(np.array(existing_dirs).astype(int))
    else:
        dirmax = 0

    # Create subfolders for each new sequence
    for sequence in range(0, no_sequences):
        seq_path = os.path.join(action_path, str(dirmax + sequence))
        if not os.path.exists(seq_path):
            os.makedirs(seq_path)

new code

In [ ]:
# import os
# import numpy as np

# # Base directory (change if needed)
# base_dir = os.getcwd()

# # Define actions
# actions = np.array(['hello', 'thank_you', 'see_you_later'])

# # Paths
# DATA_PATH = os.path.join(base_dir, 'MP_Data')
# VIDEO_PATH = os.path.join(base_dir, 'videos')

# # Create main folders if they don't exist
# os.makedirs(DATA_PATH, exist_ok=True)
# os.makedirs(VIDEO_PATH, exist_ok=True)

# # Parameters
# no_sequences = 30
# sequence_length = 30
# start_folder = 30

# # Create folder structure
# for action in actions:
#     action_path = os.path.join(DATA_PATH, action)
#     video_path = os.path.join(VIDEO_PATH, action)

#     os.makedirs(action_path, exist_ok=True)
#     os.makedirs(video_path, exist_ok=True)

#     # Create sequence folders
#     for sequence in range(start_folder, start_folder + no_sequences):
#         seq_path = os.path.join(action_path, str(sequence))
#         os.makedirs(seq_path, exist_ok=True)


In [ ]:
# print("X shape:", X.shape)
# print("y shape:", y.shape)

# # Check class distribution
# print("Hello samples:", np.sum(np.argmax(y, axis=1) == 0))
# print("Thank_you samples:", np.sum(np.argmax(y, axis=1) == 1))
# print("See_you_later samples:", np.sum(np.argmax(y, axis=1) == 2))


NameError: name 'X' is not defined

# 5. Collecting Keypoint Values for Training and Testing

Code Block 5A

In [ ]:
# Paths for original videos (where the raw sign language videos will be stored)
base_video_path = os.path.join(base_dir, 'videos')

# Create the base 'videos' directory if it doesn't exist
if not os.path.exists(base_video_path):
    os.makedirs(base_video_path)

# Loop through each action (e.g., 'hello', 'thankyou', 'seeyoulater')
for action in actions:
    # Define paths for the videos and the extracted data (keypoints)
    action_video_path = os.path.join(base_video_path, action)
    action_data_path = os.path.join(DATA_PATH, action)
    os.makedirs(action_data_path, exist_ok=True)  # Ensure the data path exists

    # Create 30 subfolders (0 to 29) under the action's data directory, one for each frame position
    for i in range(no_sequences):
        sequence_folder = os.path.join(action_data_path, str(i))
        os.makedirs(sequence_folder, exist_ok=True)

    video_counter = 0  # Keeps track of how many videos have been processed

    # Loop through all video files in the action's video folder
    for video_file in os.listdir(action_video_path):
        print(f"\n▶ Processing {video_file}")

        full_video_path = os.path.join(action_video_path, video_file)

        # Load the video using OpenCV
        cap = cv2.VideoCapture(full_video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))  # Get total number of frames
        print(f"Total frames: {total_frames}")

        # Generate 30 evenly spaced frame indices from the video
        target_frame_count = 30
        frame_indices = np.linspace(0, total_frames - 1, target_frame_count, dtype=int)

        # Set up MediaPipe Holistic model for keypoint detection
        with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
            for i, frame_index in enumerate(frame_indices):
                cap.set(cv2.CAP_PROP_POS_FRAMES, frame_index)  # Jump to the desired frame
                ret, frame = cap.read()
                if not ret:
                    print(f"⚠ Could not read frame {frame_index}")
                    continue  # Skip if frame can't be read

                # Run detection and draw landmarks
                image, results = mediapipe_detection(frame, holistic)
                draw_styled_landmarks(image, results)

                # Extract keypoints from results
                keypoints = extract_keypoints(results)

                # Save the keypoints to a .npy file in the appropriate subfolder
                npy_path = os.path.join(action_data_path, str(i), f"{video_counter}.npy")
                np.save(npy_path, keypoints)
                print(f"✅ Saved to {npy_path}")

        # Release the video capture object
        cap.release()
        video_counter += 1  # Move to next video


▶ Processing WhatsApp Video 2026-02-05 at 9.36.46 PM.mp4
Total frames: 171
✅ Saved to /content/drive/MyDrive/sign_language_detection/MP_Data/hello/0/0.npy
✅ Saved to /content/drive/MyDrive/sign_language_detection/MP_Data/hello/1/0.npy
✅ Saved to /content/drive/MyDrive/sign_language_detection/MP_Data/hello/2/0.npy
✅ Saved to /content/drive/MyDrive/sign_language_detection/MP_Data/hello/3/0.npy
✅ Saved to /content/drive/MyDrive/sign_language_detection/MP_Data/hello/4/0.npy
✅ Saved to /content/drive/MyDrive/sign_language_detection/MP_Data/hello/5/0.npy
✅ Saved to /content/drive/MyDrive/sign_language_detection/MP_Data/hello/6/0.npy
✅ Saved to /content/drive/MyDrive/sign_language_detection/MP_Data/hello/7/0.npy
✅ Saved to /content/drive/MyDrive/sign_language_detection/MP_Data/hello/8/0.npy
✅ Saved to /content/drive/MyDrive/sign_language_detection/MP_Data/hello/9/0.npy
✅ Saved to /content/drive/MyDrive/sign_language_detection/MP_Data/hello/10/0.npy
✅ Saved to /content/drive/MyDrive/sign_lang

# 6. Preprocessing Data and Creating Labels and Features

Code Block 6A

In [ ]:
# Create a dictionary that maps each action label (e.g., 'hello') to a unique integer
# This is useful for converting string labels into numeric format for model training
label_map = {label: num for num, label in enumerate(actions)}

# Display the resulting label map
label_map

{'hello': 0, 'thank_you': 1, 'see_you_later': 2}

Code Block 6B

In [ ]:
# # Initialize lists to store sequences of keypoints and their corresponding labels
# sequences, labels = [], []

# # Loop through each action (e.g., 'hello', 'thank_you', 'see_you_later')
# for action in actions:
#     # Get the list of sequence folders (0, 1, ..., 29) for this action
#     for sequence in np.array(os.listdir(os.path.join(DATA_PATH, action))).astype(int):
#         sequence_path = os.path.join(DATA_PATH, action, str(sequence))

#         # ⚠ Skip sequences that do not have enough frames
#         existing_frames = len(os.listdir(sequence_path))
#         if existing_frames < sequence_length:
#             print(f"⚠ Skipping sequence {sequence} in action {action} (only {existing_frames} frames)")
#             continue

#         window = []  # Temporary list to hold the 30 frames of one video sequence

#         # Loop through each frame in the sequence (expected to be 30 frames)
#         for frame_num in range(sequence_length):
#             frame_file = os.path.join(sequence_path, "{}.npy".format(frame_num))

#             # ⚠ If a frame is missing, fill with zeros (optional)
#             if not os.path.exists(frame_file):
#                 print(f"⚠ Frame {frame_num} missing in sequence {sequence}, filling with zeros")
#                 res = np.zeros(1662)  # Replace 1662 with the length of your keypoints vector
#             else:
#                 res = np.load(frame_file)

#             window.append(res)  # Append the keypoints to the current sequence

#         # After collecting all frames, add the full sequence to the dataset
#         sequences.append(window)

#         # Add the corresponding label (numeric) for this action
#         labels.append(label_map[action])

# # Convert the list of sequences into a NumPy array for model training
# X = np.array(sequences)

# # Convert the list of numeric labels into one-hot encoded format
# y = to_categorical(labels).astype(int)


⚠ Skipping sequence 30 in action hello (only 0 frames)
⚠ Skipping sequence 31 in action hello (only 0 frames)
⚠ Skipping sequence 32 in action hello (only 0 frames)
⚠ Skipping sequence 33 in action hello (only 0 frames)
⚠ Skipping sequence 34 in action hello (only 0 frames)
⚠ Skipping sequence 35 in action hello (only 0 frames)
⚠ Skipping sequence 36 in action hello (only 0 frames)
⚠ Skipping sequence 37 in action hello (only 0 frames)
⚠ Skipping sequence 38 in action hello (only 0 frames)
⚠ Skipping sequence 39 in action hello (only 0 frames)
⚠ Skipping sequence 40 in action hello (only 0 frames)
⚠ Skipping sequence 41 in action hello (only 0 frames)
⚠ Skipping sequence 42 in action hello (only 0 frames)
⚠ Skipping sequence 43 in action hello (only 0 frames)
⚠ Skipping sequence 44 in action hello (only 0 frames)
⚠ Skipping sequence 45 in action hello (only 0 frames)
⚠ Skipping sequence 46 in action hello (only 0 frames)
⚠ Skipping sequence 47 in action hello (only 0 frames)
⚠ Skipping

KeyboardInterrupt: 

In [ ]:
# Initialize lists to store sequences of keypoints and their corresponding labels
sequences, labels = [], []

# Loop through each action (e.g., 'hello', 'thank_you', 'see_you_later')
for action in actions:
    # Get the list of sequence folders (0, 1, ..., 29) for this action
    for sequence in np.array(os.listdir(os.path.join(DATA_PATH, action))).astype(int):
        window = []  # Temporary list to hold the 30 frames of one video sequence

        # Loop through each frame in the sequence (expected to be 30 frames)
        for frame_num in range(sequence_length):
            # Load the keypoints for this frame from its .npy file
            res = np.load(os.path.join(DATA_PATH, action, str(sequence), "{}.npy".format(frame_num)))
            window.append(res)  # Append the keypoints to the current sequence

        # After collecting all 30 frames, add the full sequence to the dataset
        sequences.append(window)

        # Add the corresponding label (numeric) for this action
        labels.append(label_map[action])

# Convert the list of sequences into a NumPy array for model training
# Each sequence is a 30-frame window, where each frame contains the extracted keypoints
X = np.array(sequences)

# Convert the list of numeric labels into one-hot encoded format
# For example, label 0 becomes [1, 0, 0], label 1 becomes [0, 1, 0], etc.
y = to_categorical(labels).astype(int)

KeyboardInterrupt: 

In [ ]:
# sequences, labels = [], []

# for action in actions:
#     action_path = os.path.join(DATA_PATH, action)

#     for sequence in sorted(os.listdir(action_path), key=int):
#         sequence_path = os.path.join(action_path, sequence)

#         frame_files = sorted(
#             [f for f in os.listdir(sequence_path) if f.endswith('.npy')],
#             key=lambda x: int(x.split('.')[0])
#         )

#         # Skip incomplete sequences
#         if len(frame_files) < sequence_length:
#             print(f"⚠ Skipping {action}/{sequence} (only {len(frame_files)} frames)")
#             continue

#         window = []
#         for frame_file in frame_files[:sequence_length]:
#             window.append(np.load(os.path.join(sequence_path, frame_file)))

#         sequences.append(window)
#         labels.append(label_map[action])

# X = np.array(sequences)
# y = to_categorical(labels).astype(int)

# print("X shape:", X.shape)
# print("y shape:", y.shape)

⚠ Skipping hello/30 (only 0 frames)
⚠ Skipping hello/31 (only 0 frames)
⚠ Skipping hello/32 (only 0 frames)
⚠ Skipping hello/33 (only 0 frames)
⚠ Skipping hello/34 (only 0 frames)
⚠ Skipping hello/35 (only 0 frames)
⚠ Skipping hello/36 (only 0 frames)
⚠ Skipping hello/37 (only 0 frames)
⚠ Skipping hello/38 (only 0 frames)
⚠ Skipping hello/39 (only 0 frames)
⚠ Skipping hello/40 (only 0 frames)
⚠ Skipping hello/41 (only 0 frames)
⚠ Skipping hello/42 (only 0 frames)
⚠ Skipping hello/43 (only 0 frames)
⚠ Skipping hello/44 (only 0 frames)
⚠ Skipping hello/45 (only 0 frames)
⚠ Skipping hello/46 (only 0 frames)
⚠ Skipping hello/47 (only 0 frames)
⚠ Skipping hello/48 (only 0 frames)
⚠ Skipping hello/49 (only 0 frames)
⚠ Skipping hello/50 (only 0 frames)
⚠ Skipping hello/51 (only 0 frames)
⚠ Skipping hello/52 (only 0 frames)
⚠ Skipping hello/53 (only 0 frames)
⚠ Skipping hello/54 (only 0 frames)
⚠ Skipping hello/55 (only 0 frames)
⚠ Skipping hello/56 (only 0 frames)
⚠ Skipping hello/57 (only 0 

In [ ]:
sequences, labels = [], []

for action in actions:
    action_path = os.path.join(DATA_PATH, action)

    # video IDs are stored as filenames inside frame 0
    video_ids = sorted(
        [f.split(".")[0] for f in os.listdir(os.path.join(action_path, "0"))]
    )

    for vid in video_ids:
        window = []

        for frame_num in range(sequence_length):
            frame_file = os.path.join(
                action_path,
                str(frame_num),
                f"{vid}.npy"
            )

            if not os.path.exists(frame_file):
                print(f"⚠ Missing: {frame_file}")
                break

            window.append(np.load(frame_file))

        if len(window) == sequence_length:
            sequences.append(window)
            labels.append(label_map[action])

X = np.array(sequences)
y = to_categorical(labels).astype(int)

print("✅ X shape:", X.shape)
print("✅ y shape:", y.shape)

✅ X shape: (90, 30, 1662)
✅ y shape: (90, 3)


In [ ]:
# for action in actions:
#     action_path = os.path.join(DATA_PATH, action)
#     print("\n", action)
#     for seq in sorted(os.listdir(action_path)):
#         seq_path = os.path.join(action_path, seq)
#         print(seq, "->", len(os.listdir(seq_path)))



 hello
0 -> 0
1 -> 0
10 -> 0
11 -> 0
12 -> 0
13 -> 0
14 -> 0
15 -> 0
16 -> 0
17 -> 0
18 -> 0
19 -> 0
2 -> 0
20 -> 0
21 -> 0
22 -> 0
23 -> 0
24 -> 0
25 -> 0
26 -> 0
27 -> 0
28 -> 0
29 -> 0
3 -> 0
30 -> 0
31 -> 0
32 -> 0
33 -> 0
34 -> 0
35 -> 0
36 -> 0
37 -> 0
38 -> 0
39 -> 0
4 -> 0
40 -> 0
41 -> 0
42 -> 0
43 -> 0
44 -> 0
45 -> 0
46 -> 0
47 -> 0
48 -> 0
49 -> 0
5 -> 0
50 -> 0
51 -> 0
52 -> 0
53 -> 0
54 -> 0
55 -> 0
56 -> 0
57 -> 0
58 -> 0
59 -> 0
6 -> 0
7 -> 0
8 -> 0
9 -> 0

 thank_you
0 -> 0
1 -> 0
10 -> 0
11 -> 0
12 -> 0
13 -> 0
14 -> 0
15 -> 0
16 -> 0
17 -> 0
18 -> 0
19 -> 0
2 -> 0
20 -> 0
21 -> 0
22 -> 0
23 -> 0
24 -> 0
25 -> 0
26 -> 0
27 -> 0
28 -> 0
29 -> 0
3 -> 0
30 -> 0
31 -> 0
32 -> 0
33 -> 0
34 -> 0
35 -> 0
36 -> 0
37 -> 0
38 -> 0
39 -> 0
4 -> 0
40 -> 0
41 -> 0
42 -> 0
43 -> 0
44 -> 0
45 -> 0
46 -> 0
47 -> 0
48 -> 0
49 -> 0
5 -> 0
50 -> 0
51 -> 0
52 -> 0
53 -> 0
54 -> 0
55 -> 0
56 -> 0
57 -> 0
58 -> 0
59 -> 0
6 -> 0
7 -> 0
8 -> 0
9 -> 0

 see_you_later
0 -> 0
1 -> 0
10 -> 0
11

In [ ]:
# for action in actions:
#     action_path = os.path.join(DATA_PATH, action)

#     for seq in os.listdir(action_path):
#         seq_path = os.path.join(action_path, seq)

#         if os.path.isdir(seq_path):
#             if len(os.listdir(seq_path)) == 0:
#                 print("Deleting empty:", seq_path)
#                 os.rmdir(seq_path)


Deleting empty: /content/MP_Data/hello/46
Deleting empty: /content/MP_Data/hello/7
Deleting empty: /content/MP_Data/hello/55
Deleting empty: /content/MP_Data/hello/33
Deleting empty: /content/MP_Data/hello/10
Deleting empty: /content/MP_Data/hello/23
Deleting empty: /content/MP_Data/hello/38
Deleting empty: /content/MP_Data/hello/12
Deleting empty: /content/MP_Data/hello/40
Deleting empty: /content/MP_Data/hello/47
Deleting empty: /content/MP_Data/hello/18
Deleting empty: /content/MP_Data/hello/57
Deleting empty: /content/MP_Data/hello/24
Deleting empty: /content/MP_Data/hello/56
Deleting empty: /content/MP_Data/hello/19
Deleting empty: /content/MP_Data/hello/36
Deleting empty: /content/MP_Data/hello/13
Deleting empty: /content/MP_Data/hello/4
Deleting empty: /content/MP_Data/hello/49
Deleting empty: /content/MP_Data/hello/34
Deleting empty: /content/MP_Data/hello/30
Deleting empty: /content/MP_Data/hello/48
Deleting empty: /content/MP_Data/hello/52
Deleting empty: /content/MP_Data/hel

Code Block 6C

In [ ]:
# Split the dataset into training and testing sets
# 95% of the data will be used for training, 5% for testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.05)

# Print the shapes of the resulting datasets to verify the split
print("X_train shape:", X_train.shape)  # Shape: (num_train_samples, num_frames, num_features)
print("X_test shape:", X_test.shape)    # Shape: (num_test_samples, num_frames, num_features)
print("y_train shape:", y_train.shape)  # Shape: (num_train_samples, num_classes)
print("y_test shape:", y_test.shape)    # Shape: (num_test_samples, num_classes)

X_train shape: (85, 30, 1662)
X_test shape: (5, 30, 1662)
y_train shape: (85, 3)
y_test shape: (5, 3)


# 7. Building and Training an LSTM Neural Network

Code Block 7A

In [ ]:
# Set the directory where TensorBoard logs will be saved
log_dir = os.path.join('Logs')

# Create a TensorBoard callback to log training metrics (loss, accuracy, etc.)
# This allows you to visualize training progress in TensorBoard
tb_callback = TensorBoard(log_dir=log_dir)

# Build a sequential neural network model
model = Sequential()

# First LSTM layer with 64 units; returns sequences to feed into next LSTM
# Input shape is (30 time steps, 1662 features per frame)
model.add(LSTM(64, return_sequences=True, activation='relu', input_shape=(30,1662)))

# Second LSTM layer with 128 units; also returns sequences
model.add(LSTM(128, return_sequences=True, activation='relu'))

# Third LSTM layer with 64 units; does not return sequences (only final output is passed on)
model.add(LSTM(64, return_sequences=False, activation='relu'))

# Fully connected dense layer with 64 units and ReLU activation
model.add(Dense(64, activation='relu'))

# Fully connected dense layer with 32 units and ReLU activation
model.add(Dense(32, activation='relu'))

# Output layer with number of units equal to number of actions (e.g., 3),
# and softmax activation for multi-class classification
model.add(Dense(actions.shape[0], activation='softmax'))




/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Code Block 7B

In [ ]:
# Compile the model with the following settings:
# 'Adam' optimizer is used, which is an adaptive learning rate method commonly used in training deep learning models
model.compile(optimizer='Adam',
              # Use 'categorical_crossentropy' as the loss function, suitable for multi-class classification problems
              loss='categorical_crossentropy',
              # Track 'categorical_accuracy' during training to evaluate how well the model is classifying the correct action
              metrics=['categorical_accuracy'])

In [ ]:
# Train the model using the training data
# X_train: Input data (sequences of keypoints)
# y_train: Target labels (one-hot encoded labels for the actions)
# epochs=200: The model will train for 200 epochs (iterations over the entire dataset)
# callbacks=[tb_callback]: This will use the TensorBoard callback to log training progress and metrics for visualization
model.fit(X_train, y_train, epochs=300, callbacks=[tb_callback])

Epoch 1/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 6s 137ms/step - categorical_accuracy: 0.2447 - loss: 1.5166
Epoch 2/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 127ms/step - categorical_accuracy: 0.3111 - loss: 2.9734
Epoch 3/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 128ms/step - categorical_accuracy: 0.3659 - loss: 1.3655
Epoch 4/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 127ms/step - categorical_accuracy: 0.3992 - loss: 1.0789
Epoch 5/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 233ms/step - categorical_accuracy: 0.4207 - loss: 1.0862
Epoch 6/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 231ms/step - categorical_accuracy: 0.6399 - loss: 0.9224
Epoch 7/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 224ms/step - categorical_accuracy: 0.3287 - loss: 1.3979
Epoch 8/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 224ms/step - categorical_accuracy: 0.4246 - loss: 1.1197
Epoch 9/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 226ms/step - categorical_accuracy: 0.4305 - loss: 1.8363
Epoch 10/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 132ms/step - categorical_accuracy: 0.4324 - loss: 1.0551
Epoch 11/300
3/3 ━━━━━━━━━━━━

# 8. Making Predictions

Code Block 8A

In [ ]:
# Use the trained model to make predictions on the test data
# X_test: Input data (test set of sequences of keypoints)
# The model predicts the output (probabilities for each class) for each sample in the test set
y_pred = model.predict(X_test)

# Display the predictions (e.g., probabilities for each class for each test sample)
y_pred.shape

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 481ms/step


(5, 3)

Code Block 8B

In [ ]:
# Get the predicted action label for the 4th sample in the test set (index 3)
# np.argmax(res[3]) returns the index of the highest predicted probability (most likely class) for the 4th sample
# The index is used to retrieve the corresponding action label from the 'actions' array
actions[np.argmax(y_pred[0])]  # TODO: Change index to see other predictions

'hello'

Code Block 8C

In [ ]:
# Get the true action label for the 4th sample in the test set (index 3)
# np.argmax(y_test[3]) returns the index of the highest value in the one-hot encoded true label for the 4th sample
# The index is used to retrieve the corresponding action label from the 'actions' array
actions[np.argmax(y_test[0])] # TODO: Change to the same index as the previous code block to see if the results match

'hello'

# 9. Saving Weights

Code Block 9A

In [ ]:
# Define the file path where the trained model will be saved
# The model will be saved as a .h5 file (HDF5 format), which stores the model architecture, weights, and optimizer state
model_path = os.path.join(base_dir, 'my_model1_new.keras')

# Save the trained model to the specified path
# This allows you to later load the model for inference or continue training
model.save(model_path)

Code Block 9B

In [ ]:
# Load the saved model from the specified file path
model_path = os.path.join(base_dir, 'my_model1_new.keras')
# 'model_path' points to the location where the model was previously saved (e.g., 'action.h5')
# The model is loaded along with its architecture, weights, and optimizer state, making it ready for inference or further training
model = load_model(model_path)

# 10. Evaluating Using a Confusion Matrix and Accuracy

Code Block 10A

In [ ]:
# Use the trained model to make predictions on the test data (X_test)
# The model outputs the predicted probabilities for each class in the test set
# y_pred will contain the predicted probabilities or class labels for each sample in X_test
y_pred = model.predict(X_test)

# Convert the one-hot encoded true labels (y_test) into class labels (indices)
# np.argmax(y_test, axis=1) finds the index of the maximum value along the second axis (for each sample)
# This index corresponds to the true class label (e.g., [1, 0, 0] -> class 0, [0, 1, 0] -> class 1)
y_true = np.argmax(y_test, axis=1).tolist()

# Convert the predicted probabilities (y_pred) into predicted class labels (indices)
# np.argmax(y_pred, axis=1) finds the index of the highest predicted probability for each sample
# This index corresponds to the predicted class label
y_pred = np.argmax(y_pred, axis=1).tolist()

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 487ms/step


Code Block 10B

In [ ]:
# Calculate multilabel confusion matrix
conf_matrix = multilabel_confusion_matrix(y_true, y_pred)

for idx, matrix in enumerate(conf_matrix):
    print(f"\nConfusion Matrix for class '{actions[idx]}':")
    df = pd.DataFrame(matrix,
                      index=["Actual Negative", "Actual Positive"],
                      columns=["Predicted Negative", "Predicted Positive"])
    print(df)




Confusion Matrix for class 'hello':
                 Predicted Negative  Predicted Positive
Actual Negative                   4                   0
Actual Positive                   0                   1

Confusion Matrix for class 'thank_you':
                 Predicted Negative  Predicted Positive
Actual Negative                   4                   0
Actual Positive                   0                   1

Confusion Matrix for class 'see_you_later':
                 Predicted Negative  Predicted Positive
Actual Negative                   2                   0
Actual Positive                   0                   3


In [ ]:
# # Calculate multilabel confusion matrix
# conf_matrix = multilabel_confusion_matrix(y_true, y_pred)

# for idx, matrix in enumerate(conf_matrix):
#     print(f"\nConfusion Matrix for class '{actions[idx]}':")
#     df = pd.DataFrame(matrix,
#                       index=["Actual Negative", "Actual Positive"],
#                       columns=["Predicted Negative", "Predicted Positive"])
#     print(df)



from sklearn.metrics import confusion_matrix
import pandas as pd

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=range(len(actions))
)

df = pd.DataFrame(
    cm,
    index=[f"Actual {a}" for a in actions],
    columns=[f"Predicted {a}" for a in actions]
)

print(df)

import numpy as np

unique, counts = np.unique(y_true, return_counts=True)
for u, c in zip(unique, counts):
    print(actions[u], c)


                      Predicted hello  Predicted thank_you  \
Actual hello                        2                    0   
Actual thank_you                    0                    2   
Actual see_you_later                0                    0   

                      Predicted see_you_later  
Actual hello                                0  
Actual thank_you                            0  
Actual see_you_later                        1  
hello 2
thank_you 2
see_you_later 1


Code Block 10C

In [ ]:
# Calculate the accuracy score to evaluate the model's performance
# The accuracy score measures the ratio of correctly predicted labels to the total number of samples
# It compares the true labels (y_true) with the predicted labels (y_pred) and returns the percentage of correct predictions
accuracy_score(y_true, y_pred)

1.0

# 11. Testing with Videos

Code Block 11A

In [ ]:
# Paths
test_video = "test_video.mp4" # TODO: Change to the name of your video
base_test_path = "/content/drive/MyDrive/sign_language_detection/test_videos"
input_video_path = os.path.join(base_test_path, test_video)
output_video_path = os.path.join(base_test_path, "output_video_latest_test111.mp4")
os.makedirs(base_test_path, exist_ok=True)

colors = [tuple(random.randint(0, 255) for _ in range(3)) for _ in actions]

def prob_viz(res, actions, input_frame, colors):
    output_frame = input_frame.copy()

    for num, prob in enumerate(res):
        color = colors[num]
        cv2.rectangle(output_frame, (0,60+num*40), (int(prob*100), 90+num*40), color, -1)
        cv2.putText(output_frame, actions[num], (0, 85+num*40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255), 2, cv2.LINE_AA)

    return output_frame


# Detection variables
sequence = []
sentence = []
predictions = []
threshold = 0.5

# Load video
cap = cv2.VideoCapture(input_video_path)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

# Output video setup
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

# Mediapipe setup
mp_holistic = mp.solutions.holistic
with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # ✅ Flip ONCE at the very beginning (fixes input video orientation)
        # frame = cv2.flip(frame, 0)

        # Detection
        image, results = mediapipe_detection(frame, holistic)
        draw_styled_landmarks(image, results)

        # Prediction logic
        keypoints = extract_keypoints(results)
        sequence.append(keypoints)
        sequence = sequence[-30:]

        if len(sequence) == 30:
            res = model.predict(np.expand_dims(sequence, axis=0))[0]
            predictions.append(np.argmax(res))

            if np.unique(predictions[-10:])[0] == np.argmax(res):
                if res[np.argmax(res)] > threshold:
                    if len(sentence) == 0 or actions[np.argmax(res)] != sentence[-1]:
                        sentence.append(actions[np.argmax(res)])

            if len(sentence) > 5:
                sentence = sentence[-5:]

            image = prob_viz(res, actions, image, colors)

        # Add label
        cv2.rectangle(image, (0, 0), (640, 40), (245, 117, 16), -1)
        cv2.putText(image, ' '.join(sentence), (3, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)

        # ✅ Save the frame (already flipped correctly)
        out.write(image)

    cap.release()
    out.release()
    cv2.destroyAllWindows()

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 101ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step
1/1 ━━━━━━━━━

In [ ]:
# Paths
test_video = "test_video.mp4" # TODO: Change to the name of your video
base_test_path = "/content/drive/MyDrive/sign_language_detection/test_videos"
input_video_path = os.path.join(base_test_path, test_video)
output_video_path = os.path.join(base_test_path, "output_video_latest_test111_2026_abb.mp4")
os.makedirs(base_test_path, exist_ok=True)

colors = [tuple(random.randint(0, 255) for _ in range(3)) for _ in actions]

def prob_viz(res, actions, input_frame, colors):
    output_frame = input_frame.copy()

    for num, prob in enumerate(res):
        color = colors[num]
        cv2.rectangle(output_frame, (0,60+num*40), (int(prob*100), 90+num*40), color, -1)
        cv2.putText(output_frame, actions[num], (0, 85+num*40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255), 2, cv2.LINE_AA)

    return output_frame


# Detection variables
sequence = []
sentence = []
predictions = []
threshold = 0.5

# Load video
cap = cv2.VideoCapture(input_video_path)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

# Output video setup
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

# Mediapipe setup
mp_holistic = mp.solutions.holistic
with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # ✅ Flip ONCE at the very beginning (fixes input video orientation)
        # frame = cv2.flip(frame, 0)

        # Detection
        image, results = mediapipe_detection(frame, holistic)
        draw_styled_landmarks(image, results)

        # Prediction logic
        keypoints = extract_keypoints(results)
        sequence.append(keypoints)
        sequence = sequence[-30:]

        if len(sequence) == 30:
            res = model.predict(np.expand_dims(sequence, axis=0))[0]
            predictions.append(np.argmax(res))

            if np.unique(predictions[-10:])[0] == np.argmax(res):
                if res[np.argmax(res)] > threshold:
                    if len(sentence) == 0 or actions[np.argmax(res)] != sentence[-1]:
                        sentence.append(actions[np.argmax(res)])

            if len(sentence) > 5:
                sentence = sentence[-5:]

            image = prob_viz(res, actions, image, colors)

        # Add label
        cv2.rectangle(image, (0, 0), (640, 40), (245, 117, 16), -1)
        cv2.putText(image, ' '.join(sentence), (3, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)

        # ✅ Save the frame (already flipped correctly)
        out.write(image)

    cap.release()
    out.release()
    cv2.destroyAllWindows()

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 143ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 212ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 142ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 144ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 103ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 141ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 142ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 120ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 130ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 93ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 100ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step
1/1 

In [ ]:
import cv2
import numpy as np
import os
import random
import mediapipe as mp

# ---------------- Paths ----------------
test_video = "test_video.mp4"
base_test_path = "/content/drive/MyDrive/sign_language_detection/test_videos"

input_video_path = os.path.join(base_test_path, test_video)
output_video_path = os.path.join(base_test_path, "output_video_latest_test111_2026_abb_new.mp4")

os.makedirs(base_test_path, exist_ok=True)

# ---------------- Visualization Colors ----------------
colors = [tuple(random.randint(0, 255) for _ in range(3)) for _ in actions]


# ---------------- Probability Visualization ----------------
def prob_viz(res, actions, input_frame, colors):
    output_frame = input_frame.copy()

    for num, prob in enumerate(res):
        color = colors[num]

        cv2.rectangle(output_frame,
                      (0, 60 + num * 40),
                      (int(prob * 100), 90 + num * 40),
                      color,
                      -1)

        cv2.putText(output_frame,
                    actions[num],
                    (0, 85 + num * 40),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1,
                    (255, 255, 255),
                    2,
                    cv2.LINE_AA)

    return output_frame


# ---------------- Detection Variables ----------------
sequence = []
predictions = []
current_action = ""     # ⭐ Only one action shown
threshold = 0.5


# ---------------- Load Video ----------------
cap = cv2.VideoCapture(input_video_path)

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)


# ---------------- Output Video Setup ----------------
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))


# ---------------- Mediapipe Setup ----------------
mp_holistic = mp.solutions.holistic

with mp_holistic.Holistic(min_detection_confidence=0.5,
                          min_tracking_confidence=0.5) as holistic:

    while cap.isOpened():
        ret, frame = cap.read()

        if not ret:
            break

        # -------- Mediapipe Detection --------
        image, results = mediapipe_detection(frame, holistic)
        draw_styled_landmarks(image, results)

        # -------- Extract Keypoints --------
        keypoints = extract_keypoints(results)
        sequence.append(keypoints)
        sequence = sequence[-30:]

        # -------- Prediction --------
        if len(sequence) == 30:

            res = model.predict(np.expand_dims(sequence, axis=0))[0]
            predictions.append(np.argmax(res))

            # ⭐ Show only current confident action
            if len(predictions) >= 10:
                if np.unique(predictions[-10:])[0] == np.argmax(res):
                    if res[np.argmax(res)] > threshold:
                        current_action = actions[np.argmax(res)]

            # -------- Probability Bars --------
            image = prob_viz(res, actions, image, colors)

        # -------- Display Current Action --------
        cv2.rectangle(image, (0, 0), (640, 40), (245, 117, 16), -1)

        label = f"{current_action}"
        cv2.putText(image,
                    label,
                    (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1,
                    (255, 255, 255),
                    2,
                    cv2.LINE_AA)

        # -------- Save Frame --------
        out.write(image)

    cap.release()
    out.release()
    cv2.destroyAllWindows()

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 93ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 119ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 103ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 104ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━

In [ ]:
import cv2
import numpy as np
import random
import mediapipe as mp
from tensorflow.keras.models import load_model

# ---------------- Load Model and Define Actions ----------------
model = load_model("/content/drive/MyDrive/sign_language_detection/my_model1_new.keras")
actions = ['hello', 'thank_you', 'see_you_later']

# ---------------- Visualization Colors ----------------
colors = [tuple(random.randint(0, 255) for _ in range(3)) for _ in actions]

# ---------------- Helper Functions ----------------

def mediapipe_detection(image, model):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image.flags.writeable = False
    results = model.process(image)
    image.flags.writeable = True
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    return image, results

def extract_keypoints(results):
    pose = np.array([[res.x, res.y, res.z, res.visibility]
                     for res in results.pose_landmarks.landmark]).flatten() \
        if results.pose_landmarks else np.zeros(33*4)

    face = np.array([[res.x, res.y, res.z]
                     for res in results.face_landmarks.landmark]).flatten() \
        if results.face_landmarks else np.zeros(468*3)

    lh = np.array([[res.x, res.y, res.z]
                   for res in results.left_hand_landmarks.landmark]).flatten() \
        if results.left_hand_landmarks else np.zeros(21*3)

    rh = np.array([[res.x, res.y, res.z]
                   for res in results.right_hand_landmarks.landmark]).flatten() \
        if results.right_hand_landmarks else np.zeros(21*3)

    return np.concatenate([pose, face, lh, rh])

def draw_styled_landmarks(image, results):
    mp_drawing.draw_landmarks(
        image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS)
    mp_drawing.draw_landmarks(
        image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS)
    mp_drawing.draw_landmarks(
        image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS)

def prob_viz(res, actions, input_frame, colors):
    output_frame = input_frame.copy()
    for num, prob in enumerate(res):
        color = colors[num]
        cv2.rectangle(output_frame,
                      (0, 60 + num * 40),
                      (int(prob * 640), 90 + num * 40),
                      color,
                      -1)
        cv2.putText(output_frame,
                    actions[num],
                    (0, 85 + num * 40),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1,
                    (255, 255, 255),
                    2,
                    cv2.LINE_AA)
    return output_frame

# ---------------- Detection Variables ----------------
sequence = []
predictions = []
current_action = ""
threshold = 0.5

# ---------------- Mediapipe Setup ----------------
mp_holistic = mp.solutions.holistic
mp_drawing = mp.solutions.drawing_utils

# ---------------- Real-time Webcam Capture ----------------
cap = cv2.VideoCapture(0)  # 0 = default webcam

with mp_holistic.Holistic(min_detection_confidence=0.5,
                          min_tracking_confidence=0.5) as holistic:

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            continue

        # Mediapipe detection
        image, results = mediapipe_detection(frame, holistic)
        draw_styled_landmarks(image, results)

        # Extract keypoints and store sequence
        keypoints = extract_keypoints(results)
        sequence.append(keypoints)
        sequence = sequence[-30:]

        # Prediction
        if len(sequence) == 30:
            res = model.predict(np.expand_dims(sequence, axis=0), verbose=0)[0]
            predictions.append(np.argmax(res))

            # Show only current confident action
            if len(predictions) >= 10:
                if np.unique(predictions[-10:])[0] == np.argmax(res):
                    if res[np.argmax(res)] > threshold:
                        current_action = actions[np.argmax(res)]

            # Probability bars
            image = prob_viz(res, actions, image, colors)

        # Display current action
        cv2.rectangle(image, (0, 0), (640, 40), (245, 117, 16), -1)
        cv2.putText(image,
                    current_action,
                    (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1,
                    (255, 255, 255),
                    2,
                    cv2.LINE_AA)

        # Show webcam feed
        cv2.imshow("Real-time Sign Language Detection", image)

        # Exit on pressing 'q'
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

In [ ]:
# ===============================
# IMPORTS
# ===============================
import os
import cv2
import numpy as np
import mediapipe as mp
import random
import pyttsx3
import tensorflow as tf

# ===============================
# LOAD TRAINED MODEL
# ===============================
model = tf.keras.models.load_model("model")  # change if needed

# ===============================
# ACTIONS (ORDER MUST MATCH TRAINING)
# ===============================
actions = np.array(["hello", "thank_you", "see_you_later"])

# ===============================
# FIXED COLORS (BGR for OpenCV)
# ===============================
action_colors = {
    "hello": (0, 255, 0),        # Green
    "thank_you": (255, 0, 0),    # Blue
    "see_you_later": (0, 0, 255) # Red
}

# ===============================
# TEXT TO SPEECH
# ===============================
engine = pyttsx3.init()
engine.setProperty("rate", 150)

def speak(text):
    engine.say(text)
    engine.runAndWait()

# ===============================
# MEDIAPIPE SETUP
# ===============================
mp_holistic = mp.solutions.holistic
mp_drawing = mp.solutions.drawing_utils

def mediapipe_detection(image, model):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image.flags.writeable = False
    results = model.process(image)
    image.flags.writeable = True
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    return image, results

def draw_styled_landmarks(image, results):
    mp_drawing.draw_landmarks(
        image, results.face_landmarks, mp_holistic.FACEMESH_CONTOURS
    )
    mp_drawing.draw_landmarks(
        image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS
    )
    mp_drawing.draw_landmarks(
        image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS
    )
    mp_drawing.draw_landmarks(
        image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS
    )

# ===============================
# KEYPOINT EXTRACTION
# ===============================
def extract_keypoints(results):
    pose = np.array([[res.x, res.y, res.z, res.visibility]
                     for res in results.pose_landmarks.landmark]).flatten() \
           if results.pose_landmarks else np.zeros(33 * 4)

    face = np.array([[res.x, res.y, res.z]
                     for res in results.face_landmarks.landmark]).flatten() \
           if results.face_landmarks else np.zeros(468 * 3)

    lh = np.array([[res.x, res.y, res.z]
                   for res in results.left_hand_landmarks.landmark]).flatten() \
         if results.left_hand_landmarks else np.zeros(21 * 3)

    rh = np.array([[res.x, res.y, res.z]
                   for res in results.right_hand_landmarks.landmark]).flatten() \
         if results.right_hand_landmarks else np.zeros(21 * 3)

    return np.concatenate([pose, face, lh, rh])

# ===============================
# PROBABILITY VISUALIZATION
# ===============================
def prob_viz(res, actions, frame):
    output = frame.copy()
    for i, prob in enumerate(res):
        action = actions[i]
        color = action_colors[action]

        cv2.rectangle(
            output,
            (0, 60 + i * 40),
            (int(prob * 100), 90 + i * 40),
            color,
            -1
        )

        cv2.putText(
            output,
            f"{action} {prob:.2f}",
            (5, 85 + i * 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (255, 255, 255),
            2
        )
    return output

# ===============================
# VIDEO PATHS
# ===============================
test_video = "test_video.mp4"   # 👈 change video name
base_path = "/content/drive/MyDrive/sign_language_detection/test_videos"

input_video_path = os.path.join(base_path, test_video)
output_video_path = os.path.join(base_path, "output_with_speech.mp4")
os.makedirs(base_path, exist_ok=True)

# ===============================
# VIDEO SETUP
# ===============================
cap = cv2.VideoCapture(input_video_path)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

# ===============================
# DETECTION VARIABLES
# ===============================
sequence = []
predictions = []
sentence = []
threshold = 0.5
last_spoken = ""

# ===============================
# MAIN LOOP
# ===============================
with mp_holistic.Holistic(
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
) as holistic:

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        image, results = mediapipe_detection(frame, holistic)
        draw_styled_landmarks(image, results)

        keypoints = extract_keypoints(results)
        sequence.append(keypoints)
        sequence = sequence[-30:]

        detected_action = None
        detected_color = (40, 40, 40)

        if len(sequence) == 30:
            res = model.predict(np.expand_dims(sequence, axis=0), verbose=0)[0]
            predictions.append(np.argmax(res))

            if np.unique(predictions[-10:])[0] == np.argmax(res):
                if res[np.argmax(res)] > threshold:
                    detected_action = actions[np.argmax(res)]
                    detected_color = action_colors[detected_action]

                    if len(sentence) == 0 or detected_action != sentence[-1]:
                        sentence.append(detected_action)

                        if detected_action != last_spoken:
                            speak(detected_action)
                            last_spoken = detected_action

            image = prob_viz(res, actions, image)

        # 🔝 TOP BAR (ONLY ONE WORD)
        cv2.rectangle(image, (0, 0), (width, 50), detected_color, -1)

        if detected_action:
            cv2.putText(
                image,
                detected_action.upper(),
                (20, 35),
                cv2.FONT_HERSHEY_SIMPLEX,
                1.3,
                (255, 255, 255),
                3
            )

        out.write(image)

    cap.release()
    out.release()
    cv2.destroyAllWindows()

print("✅ Output saved at:", output_video_path)

ValueError: File format not supported: filepath=model. Keras 3 only supports V3 `.keras` files and legacy H5 format files (`.h5` extension). Note that the legacy SavedModel format is not supported by `load_model()` in Keras 3. In order to reload a TensorFlow SavedModel as an inference-only layer in Keras 3, use `keras.layers.TFSMLayer(model, call_endpoint='serving_default')` (note that your `call_endpoint` might have a different name).

In [ ]:
!pip install pyttsx3

In [ ]:
import os
import cv2
import numpy as np
import random
import mediapipe as mp
from tensorflow.keras.models import load_model

# Load trained model
model = load_model("action.h5")

# Define actions
actions = ['hello', 'thank_you', 'see_you_later']

# Paths
test_video = "new1.mp4"  # change if needed
base_test_path = "/content/drive/MyDrive/sign_language_detection/test_videos"
input_video_path = os.path.join(base_test_path, test_video)
output_video_path = os.path.join(base_test_path, "output_video_latest11.mp4")
os.makedirs(base_test_path, exist_ok=True)

# Mediapipe setup
mp_holistic = mp.solutions.holistic
mp_drawing = mp.solutions.drawing_utils


# ---------- Helper functions ----------

def mediapipe_detection(image, model):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image.flags.writeable = False
    results = model.process(image)
    image.flags.writeable = True
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    return image, results


def extract_keypoints(results):
    pose = np.array([[res.x, res.y, res.z, res.visibility]
                     for res in results.pose_landmarks.landmark]).flatten() \
        if results.pose_landmarks else np.zeros(33*4)

    face = np.array([[res.x, res.y, res.z]
                     for res in results.face_landmarks.landmark]).flatten() \
        if results.face_landmarks else np.zeros(468*3)

    lh = np.array([[res.x, res.y, res.z]
                   for res in results.left_hand_landmarks.landmark]).flatten() \
        if results.left_hand_landmarks else np.zeros(21*3)

    rh = np.array([[res.x, res.y, res.z]
                   for res in results.right_hand_landmarks.landmark]).flatten() \
        if results.right_hand_landmarks else np.zeros(21*3)

    return np.concatenate([pose, face, lh, rh])


def draw_styled_landmarks(image, results):
    mp_drawing.draw_landmarks(
        image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS)
    mp_drawing.draw_landmarks(
        image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS)
    mp_drawing.draw_landmarks(
        image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS)


def show_prediction(res, actions, input_frame, threshold=0.5):
    output_frame = input_frame.copy()

    max_index = np.argmax(res)
    max_prob = res[max_index]

    if max_prob > threshold:
        label = actions[max_index]
    else:
        label = "No sign"

    # Top label box
    cv2.rectangle(output_frame, (0, 0), (640, 60), (0, 0, 0), -1)
    cv2.putText(output_frame, f"{label} ({int(max_prob*100)}%)",
                (10, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 1,
                (0, 255, 0), 2, cv2.LINE_AA)

    # Confidence bar
    bar_width = int(max_prob * 640)
    cv2.rectangle(output_frame, (0, 60), (bar_width, 80), (0, 255, 0), -1)
    cv2.rectangle(output_frame, (0, 60), (640, 80), (255, 255, 255), 2)

    return output_frame


# ---------- Detection variables ----------
sequence = []
threshold = 0.5

# Load video
cap = cv2.VideoCapture(input_video_path)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

# Output video
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

# ---------- Main loop ----------
with mp_holistic.Holistic(
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5) as holistic:

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Mediapipe detection
        image, results = mediapipe_detection(frame, holistic)
        draw_styled_landmarks(image, results)

        # Prediction
        keypoints = extract_keypoints(results)
        sequence.append(keypoints)
        sequence = sequence[-30:]

        if len(sequence) == 30:
            res = model.predict(np.expand_dims(sequence, axis=0), verbose=0)[0]
            image = show_prediction(res, actions, image, threshold)

        # Save frame
        out.write(image)

    cap.release()
    out.release()
    cv2.destroyAllWindows()

print("Output video saved at:", output_video_path)


ModuleNotFoundError: No module named 'mediapipe'

In [ ]:
import cv2
import numpy as np
import mediapipe as mp
from tensorflow.keras.models import load_model

# Load trained model
model = load_model("action.h5")

# Define actions
actions = ['hello', 'thank_you', 'see_you_later']

# Mediapipe setup
mp_holistic = mp.solutions.holistic
mp_drawing = mp.solutions.drawing_utils

# ---------- Helper functions ----------

def mediapipe_detection(image, model):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image.flags.writeable = False
    results = model.process(image)
    image.flags.writeable = True
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    return image, results

def extract_keypoints(results):
    pose = np.array([[res.x, res.y, res.z, res.visibility]
                     for res in results.pose_landmarks.landmark]).flatten() \
        if results.pose_landmarks else np.zeros(33*4)

    face = np.array([[res.x, res.y, res.z]
                     for res in results.face_landmarks.landmark]).flatten() \
        if results.face_landmarks else np.zeros(468*3)

    lh = np.array([[res.x, res.y, res.z]
                   for res in results.left_hand_landmarks.landmark]).flatten() \
        if results.left_hand_landmarks else np.zeros(21*3)

    rh = np.array([[res.x, res.y, res.z]
                   for res in results.right_hand_landmarks.landmark]).flatten() \
        if results.right_hand_landmarks else np.zeros(21*3)

    return np.concatenate([pose, face, lh, rh])

def draw_styled_landmarks(image, results):
    mp_drawing.draw_landmarks(
        image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS)
    mp_drawing.draw_landmarks(
        image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS)
    mp_drawing.draw_landmarks(
        image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS)

def show_prediction(res, actions, input_frame, threshold=0.5):
    output_frame = input_frame.copy()
    max_index = np.argmax(res)
    max_prob = res[max_index]

    if max_prob > threshold:
        label = actions[max_index]
    else:
        label = "No sign"

    # Top label box
    cv2.rectangle(output_frame, (0, 0), (640, 60), (0, 0, 0), -1)
    cv2.putText(output_frame, f"{label} ({int(max_prob*100)}%)",
                (10, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 1,
                (0, 255, 0), 2, cv2.LINE_AA)

    # Confidence bar
    bar_width = int(max_prob * 640)
    cv2.rectangle(output_frame, (0, 60), (bar_width, 80), (0, 255, 0), -1)
    cv2.rectangle(output_frame, (0, 60), (640, 80), (255, 255, 255), 2)

    return output_frame

# ---------- Real-time detection ----------
sequence = []
threshold = 0.5

cap = cv2.VideoCapture(0)  # 0 for default webcam

with mp_holistic.Holistic(min_detection_confidence=0.5,
                          min_tracking_confidence=0.5) as holistic:

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            print("Ignoring empty frame.")
            continue

        # Mediapipe detection
        image, results = mediapipe_detection(frame, holistic)
        draw_styled_landmarks(image, results)

        # Prediction
        keypoints = extract_keypoints(results)
        sequence.append(keypoints)
        sequence = sequence[-30:]  # last 30 frames

        if len(sequence) == 30:
            res = model.predict(np.expand_dims(sequence, axis=0), verbose=0)[0]
            image = show_prediction(res, actions, image, threshold)

        # Display
        cv2.imshow('Real-time Sign Language Detection', image)

        # Exit with 'q'
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

ModuleNotFoundError: No module named 'mediapipe'

In [ ]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 30, 64)         │       442,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 30, 128)        │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,790,027 (6.83 MB)

 Trainable params: 596,675 (2.28 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 1,193,352 (4.55 MB)

UI CODE


In [ ]:
!pip install streamlit opencv-python tensorflow mediapipe pyngrok

  Using cached numpy-2.4.2-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached numpy-2.1.3-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (62 kB)
INFO: pip is looking at multiple versions of mediapipe to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 90.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 11.2 MB/s eta 0:00:00
Using cached numpy-2.1.3-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (16.0 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 35.4 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
  Attempting uninstall: cachetools
    Found existing installation: cachetools 7.0.0
  

In [ ]:
%%writefile app.py
import streamlit as st
import tensorflow as tf
import numpy as np
import cv2
import os

# ===== Load Model =====
base_dir = "/content/drive/MyDrive/sign_language_detection"
model_path = os.path.join(base_dir, "my_model1_new.keras")

model = tf.keras.models.load_model(model_path)

st.title("Sign Language Recognition")

uploaded_file = st.file_uploader("Upload Image", type=["jpg","png","jpeg"])

if uploaded_file is not None:

    # Read uploaded image
    file_bytes = np.asarray(bytearray(uploaded_file.read()), dtype=np.uint8)
    frame = cv2.imdecode(file_bytes, 1)

    st.image(frame, channels="BGR")

    # ===== Preprocess =====
    img = cv2.resize(frame, (224,224))
    img = img / 255.0
    img = np.expand_dims(img, axis=0)

    # ===== Prediction =====
    pred = model.predict(img)
    result = str(np.argmax(pred))

    st.success(f"Prediction: {result}")

Writing app.py


In [ ]:
from pyngrok import ngrok
!streamlit run app.py &

public_url = ngrok.connect(8501)
print(public_url)




  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.67.84.24:8501

  Stopping...


ERROR:pyngrok.process.ngrok:t=2026-02-06T14:51:32+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-02-06T14:51:32+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-02-06T14:51:32+0000 lvl=eror msg="terminating with error" obj=app err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your aut

PyngrokNgrokError: The ngrok process errored on start: authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n.

In [ ]:
!pkill streamlit

In [ ]:
!ngrok config add-authtoken 39IiA4lXTkgK2trxap2a3pEYnhi_3M8qU2gKQgKqKuqxwi1Gk

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [ ]:
from pyngrok import ngrok
import time

get_ipython().system_raw('streamlit run app.py --server.port 8501 &')

time.sleep(5)

public_url = ngrok.connect(8501)
print(public_url)

NgrokTunnel: "https://truthful-hippiest-stephnie.ngrok-free.dev" -> "http://localhost:8501"


In [ ]:
from google.colab import files
files.download("/content/drive/MyDrive/sign_language_detection/my_model1_new.keras")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>